# Deep Learning Approach to Digital Image Correlation - Results Analysis

This notebook presents the key results and performance analysis of the neural network approach to Digital Image Correlation (DIC).

## Overview

This project explores using deep convolutional neural networks to perform Digital Image Correlation on synthetic speckle pattern images. The approach uses:
- Synthetic dataset generation with controlled deformations
- CNN architectures with residual and dense connections
- Training on displacement and strain field predictions
- Comparison with traditional DIC methods

## Setup and Imports

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import math

# Set plotting style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Training Performance Comparison

This section shows the training loss curves for different network architectures and configurations.

In [ ]:
# Helper function to plot training results
def plot_training_curves(results_dict, title="Training Loss Curves"):
    """
    Plot training loss curves for multiple models.
    
    Args:
        results_dict: Dictionary mapping model names to their training results
        title: Plot title
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    
    for i, (name, label) in enumerate(results_dict.items()):
        # Note: In actual usage, you would load the training data here
        # This is a placeholder showing the structure
        # data = load_trainer_results(path_proj, name)
        # loss_tr = data['loss']['train']
        # loss_ts = data['loss']['test']
        # ax.semilogy(loss_tr, color=colors[i], label=label)
        # if len(loss_ts) > 0:
        #     ax.plot(loss_ts, '--', color=colors[i])
        pass
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title(title)
    ax.legend(loc='upper right')
    ax.grid(True, which='both', alpha=0.3)
    
    # Add legend for train/validation
    handles = []
    handles.append(plt.Line2D([0], [0], color='gray', linestyle='-', label='Train'))
    handles.append(plt.Line2D([0], [0], color='gray', linestyle='--', label='Validation'))
    ax.legend(handles=handles, loc='lower left')
    
    plt.tight_layout()
    return fig, ax

### Architecture Evolution

The following models show the evolution of the network architecture:

1. **Initial model**: Baseline CNN with batch normalization
2. **Group Normalization**: Improved with group normalization for better training stability
3. **Residual Units**: Added residual connections with Mish activation
4. **Dense Connections**: Incorporated dense blocks for better gradient flow
5. **Separable Convolutions**: Final architecture with separable convolutions for efficiency

In [ ]:
# Example: Architecture comparison
# Note: Actual training data would need to be loaded from saved results

architecture_comparison = {
    'initial': 'Initial Model (Batch Norm)',
    'GroupNorm-4': 'Group Norm (4 groups)',
    'Mish-GN-4-Batch-1': 'Residual + Mish + GN',
    'denseblock/displacement': 'Dense Blocks',
    'SepConvComb-denseblock': 'Final (Sep Conv + Dense)'
}

# fig, ax = plot_training_curves(architecture_comparison, "Network Architecture Evolution")
print("Architecture comparison configurations loaded.")
print("Models tested:", list(architecture_comparison.values()))

## 2. Displacement Field Predictions

Visual comparison of predicted vs. target displacement fields.

In [ ]:
def plot_displacement_comparison(prediction, target, title="Displacement Field Comparison"):
    """
    Plot side-by-side comparison of predicted and target displacement fields.
    
    Args:
        prediction: Predicted displacement field [2, H, W] for (u, v)
        target: Ground truth displacement field [2, H, W]
        title: Plot title
    """
    fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(12, 15))
    
    labels = ['u (horizontal)', 'v (vertical)']
    
    for i in range(2):
        vmax = max(abs(target[i]).max(), abs(prediction[i]).max())
        
        # Target
        im1 = axs[i, 0].imshow(target[i], cmap='jet', vmin=-vmax, vmax=vmax)
        axs[i, 0].set_title(f'Target {labels[i]}')
        axs[i, 0].axis('off')
        plt.colorbar(im1, ax=axs[i, 0], fraction=0.046)
        
        # Prediction
        im2 = axs[i, 1].imshow(prediction[i], cmap='jet', vmin=-vmax, vmax=vmax)
        axs[i, 1].set_title(f'Prediction {labels[i]}')
        axs[i, 1].axis('off')
        plt.colorbar(im2, ax=axs[i, 1], fraction=0.046)
        
        # Difference
        diff = prediction[i] - target[i]
        vmax_diff = abs(diff).max()
        im3 = axs[2, i].imshow(diff, cmap='jet', vmin=-vmax_diff, vmax=vmax_diff)
        axs[2, i].set_title(f'Difference {labels[i]}')
        axs[2, i].axis('off')
        plt.colorbar(im3, ax=axs[2, i], fraction=0.046)
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

## 3. Quantitative Performance Metrics

Comparison of error statistics across different methods and element sizes.

In [ ]:
# Performance comparison data (from graphs.ipynb)
# These values represent displacement error in pixels x 1000

element_sizes = [5, 9, 17, 33, 65]  # pixels

# Dummy baseline (no learning)
dummy_avg = np.array([0.183992, 0.189235, 0.389536, 0.576842, 0.648282])
dummy_25 = np.array([0.078517, 0.084398, 0.173511, 0.261439, 0.159488])
dummy_75 = np.array([0.27625, 0.282373, 0.587807, 0.871714, 1.11464])

# Traditional DIC
dic_avg = np.array([0.171726, 0.114339, 0.066242, 0.021934, 0.007845])
dic_25 = np.array([0.069055, 0.042942, 0.022359, 0.007894, 0.003088])
dic_75 = np.array([0.252212, 0.164915, 0.089963, 0.030428, 0.011149])

# Deep-DIC (reference method)
deepdic_avg = np.array([0.239258, 0.218837, 0.215781, 0.254139, 0.192059])
deepdic_25 = np.array([0.097679, 0.088836, 0.082896, 0.106767, 0.083764])
deepdic_75 = np.array([0.355026, 0.324183, 0.311331, 0.373111, 0.266849])

# Proposed method (this work)
proposed_avg = np.array([0.030828, 0.012258, 0.011591, 0.005683, 0.005293])
proposed_25 = np.array([0.01047, 0.00465, 0.004251, 0.002044, 0.001934])
proposed_75 = np.array([0.04129, 0.01686, 0.016613, 0.007824, 0.007371])

In [ ]:
# Plot error comparison
fig, ax = plt.subplots(figsize=(10, 6))

colors = [(0.8, 0.6, 0), (1, 0, 0), 'b', 'g']
methods = [
    ('Dummy', dummy_avg, dummy_25, dummy_75),
    ('Traditional DIC', dic_avg, dic_25, dic_75),
    ('Deep-DIC', deepdic_avg, deepdic_25, deepdic_75),
    ('Proposed (This Work)', proposed_avg, proposed_25, proposed_75)
]

for i, (label, avg, p25, p75) in enumerate(methods):
    ax.semilogy(avg * 1e3, color=colors[i], label=label, linewidth=2)
    ax.semilogy(p25 * 1e3, '-.', color=colors[i], alpha=0.6)
    ax.semilogy(p75 * 1e3, '--', color=colors[i], alpha=0.6)

ax.set_xticks(range(len(element_sizes)))
ax.set_xticklabels(element_sizes)
ax.set_xlabel('Element Size (pixels)', fontsize=12)
ax.set_ylabel('Error (pixels / 1000)', fontsize=12)
ax.set_title('Displacement Error Comparison Across Methods', fontsize=14, fontweight='bold')
ax.set_ylim([1, 1200])
ax.grid(True, which='both', alpha=0.3)

# Main legend for methods
leg1 = ax.legend(loc='upper right', fontsize=10)

# Secondary legend for line styles
handles = [
    plt.Line2D([0], [0], color='gray', linestyle='-.', label='25th percentile'),
    plt.Line2D([0], [0], color='gray', linestyle='-', label='Mean'),
    plt.Line2D([0], [0], color='gray', linestyle='--', label='75th percentile')
]
leg2 = ax.legend(handles=handles, loc='lower left', fontsize=10)
ax.add_artist(leg1)

plt.tight_layout()
plt.show()

print("\nKey Findings:")
print("- Proposed method achieves significantly lower error than Deep-DIC")
print("- Performance is competitive with traditional DIC at larger element sizes")
print("- Best performance at element sizes >= 17 pixels")

## 4. Performance Statistics Summary

In [ ]:
import pandas as pd

# Create summary table
summary_data = []
for elem_size in element_sizes:
    idx = element_sizes.index(elem_size)
    summary_data.append({
        'Element Size': elem_size,
        'DIC Mean': f"{dic_avg[idx]*1e3:.2f}",
        'Deep-DIC Mean': f"{deepdic_avg[idx]*1e3:.2f}",
        'Proposed Mean': f"{proposed_avg[idx]*1e3:.2f}",
        'Improvement vs Deep-DIC': f"{(1 - proposed_avg[idx]/deepdic_avg[idx])*100:.1f}%"
    })

df = pd.DataFrame(summary_data)
print("\nDisplacement Error Summary (pixels / 1000):")
print(df.to_string(index=False))

## 5. Real Data Application: Beam Vibration Tracking

Application to real experimental data: tracking displacement in a vibrating beam.

In [ ]:
def plot_tracking_results(signal_nn, signal_dic, title="Displacement Tracking Comparison"):
    """
    Plot time-domain and frequency-domain comparison of tracking results.
    
    Args:
        signal_nn: Neural network tracking signal
        signal_dic: Traditional DIC tracking signal
        title: Plot title
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Time domain
    ax1.plot(signal_nn * 1000, label='Neural Network', linewidth=1.5)
    ax1.plot(signal_dic * 1000, label='Traditional DIC', linewidth=1.5, alpha=0.8)
    ax1.set_xlabel('Frame Number')
    ax1.set_ylabel('Displacement (pixels / 1000)')
    ax1.set_title('Time Domain Tracking')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim([0, 400])
    
    # Frequency domain
    nfft = len(signal_dic)
    x = np.arange(len(signal_nn))
    window = np.exp(x**2 * (np.log(0.001) / x[-1]**2))
    
    fft_nn = np.fft.rfft((signal_nn - signal_nn.mean()) * window[:len(signal_nn)], nfft)
    fft_dic = np.fft.rfft((signal_dic - signal_dic.mean()) * window, nfft)
    freqs = np.fft.rfftfreq(nfft)
    
    ax2.semilogy(freqs, np.abs(fft_nn), label='Neural Network', linewidth=1.5)
    ax2.semilogy(freqs, np.abs(fft_dic), label='Traditional DIC', linewidth=1.5, alpha=0.8)
    ax2.set_xlabel('Normalized Frequency')
    ax2.set_ylabel('FFT Magnitude')
    ax2.set_title('Frequency Domain Analysis')
    ax2.legend()
    ax2.grid(True, which='both', alpha=0.3)
    ax2.set_xlim([0, 0.125])
    ax2.set_ylim([1e-1, 400])
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

### Processing Speed Comparison

Computational performance metrics for real-time application.

In [ ]:
# Performance metrics from training notebooks
performance_metrics = {
    'Model': ['v0 (0 iterations)', 'v0 (1 iteration)', 'v0 (2 iterations)'],
    'Images Processed': [20000, 20000, 20000],
    'Time per Image (ms)': [26.7, 38.7, 38.7],
    'Images per Second': [37.5, 25.8, 25.8]
}

perf_df = pd.DataFrame(performance_metrics)
print("\nProcessing Speed Performance:")
print(perf_df.to_string(index=False))
print("\nNote: All measurements on NVIDIA GeForce RTX 3070 Laptop GPU")

## 6. Key Conclusions

### Achievements:

1. **Accuracy**: The proposed neural network approach achieves displacement errors 5-10x lower than Deep-DIC across most element sizes

2. **Architecture**: Combination of dense blocks, separable convolutions, and group normalization provides optimal performance

3. **Real Data**: Successfully applied to real experimental data (vibrating beam) with competitive performance to traditional DIC

4. **Speed**: Processing rates of 25-37 images/second enable near real-time analysis

### Advantages:
- No need for iterative optimization (direct prediction)
- Consistent performance across different deformation magnitudes
- GPU acceleration for parallel processing

### Limitations:
- Requires training on synthetic data
- Performance depends on similarity between training and test conditions
- Best results at element sizes >= 17 pixels

### Future Work:
- Extension to strain field prediction
- Domain adaptation for different speckle patterns
- Integration with experimental setups for online processing

## References

This work builds upon:
- Traditional Digital Image Correlation methods
- Deep-DIC: Deep learning-based digital image correlation
- Modern CNN architectures (ResNet, DenseNet)
- Efficient convolution operations (Separable Convolutions)